# Add Metadata to Scan CSV Files

This notebook maps metadata columns from `MP_Metadata_IK.csv` into individual scan CSV files.

**Metadata columns to add:**
- Batch number
- Type of cancer
- Main cancer Grouping
- Category

In [1]:
import os
import pandas as pd
import glob
from pathlib import Path

# Set working directory
work_dir = '/vast/projects/SOLACE2/imalki/Imalki_July2022/MP_project'
os.chdir(work_dir)
print(f"Working directory: {os.getcwd()}")

Working directory: /vast/projects/SOLACE2/imalki/Imalki_July2022/MP_project


In [2]:
# Read metadata file with proper encoding
metadata_df = pd.read_csv('MP_Metadata_IK.csv', encoding='latin-1')
print(f"Metadata shape: {metadata_df.shape}")
print(f"\nMetadata columns: {metadata_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(metadata_df.head())

Metadata shape: (42, 8)

Metadata columns: ['Batch number', 'Sample ID', 'Type of cancer', 'Main cancer grouping', 'Category', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']

First few rows:
  Batch number  Sample ID                     Type of cancer  \
0      21-3120   01036-1D                      Mucinous Lung   
1      21-3120  01041-2.1                     HGS Peritoneum   
2      21-3120   01072-1A                       Renal cancer   
3      21-3120   01082-1B               Renal cell carcinoma   
4      21-3120   01082-1H  Breast Invasive Ductal Carcinoma    

                    Main cancer grouping Category  Unnamed: 5  Unnamed: 6  \
0         Lung/thoracic/mesothelioma/H&N      3MP         NaN         NaN   
1                                   HGSC      2+1         NaN         NaN   
2  ï¿½GU (renal + urothelial + prostate)      3MP         NaN         NaN   
3  ï¿½GU (renal + urothelial + prostate)      3MP         NaN         NaN   
4                                 Breast    

In [3]:
# Create a lookup dictionary from metadata using Sample ID as key
metadata_lookup = {}

for idx, row in metadata_df.iterrows():
    sample_id = str(row['Sample ID']).strip()
    if sample_id and sample_id != '':
        metadata_lookup[sample_id] = {
            'Batch number': row['Batch number'],
            'Type of cancer': row['Type of cancer'],
            'Main cancer grouping': row['Main cancer grouping'],
            'Category': row['Category']
        }

print(f"Metadata lookup entries: {len(metadata_lookup)}")
print(f"\nSample of lookup:")
for k, v in list(metadata_lookup.items())[:3]:
    print(f"  {k}: {v}")

Metadata lookup entries: 41

Sample of lookup:
  01036-1D: {'Batch number': '21-3120', 'Type of cancer': 'Mucinous Lung', 'Main cancer grouping': 'Lung/thoracic/mesothelioma/H&N', 'Category': '3MP'}
  01041-2.1: {'Batch number': '21-3120', 'Type of cancer': 'HGS Peritoneum', 'Main cancer grouping': 'HGSC', 'Category': '2+1'}
  01072-1A: {'Batch number': '21-3120', 'Type of cancer': 'Renal cancer', 'Main cancer grouping': 'ï¿½GU (renal + urothelial + prostate)', 'Category': '3MP'}


In [4]:
# Debug: Show all sample IDs in metadata lookup
print("All Sample IDs in metadata:")
print(sorted(metadata_lookup.keys()))

All Sample IDs in metadata:
['01036-1D', '01041-2.1', '01059-17B', '01059-A', '01064-1.10', '01064-1B', '01064-A1', '01072-1A', '01082-1B', '01082-1H', '01085-1M', '01095-1K', '01096-08T', '01096-17T', '01096-1C', '01102-Scan2', '01109-1.8', '01109-1A', '01111-1C', '01124-2A', '01126-1.1', '01128-1.1', '01133-1B', '01134-Scan4', '01138-Scan1', '01220-Scan2', '01243-1A', '01243-1B', '01244-1D', '01252-10', '01257-1B', '01257-1C', '01264-1A', '01264_3H', '01369_Scan1', '01385-3B', '01410-1F', '01446-1N', '01446-Scan2', '01495-1K', 'nan']


In [5]:
# Find all scan CSV files and their matching metadata
# Use broader glob to catch all .ome.tif.csv files
scan_files = sorted([f for f in glob.glob('*.ome.tif.csv')])
print(f"Found {len(scan_files)} scan CSV files\n")

# Build mapping of scan files to sample IDs
file_to_sample = {}
matched = 0
not_matched = []

def extract_sample_ids(scan_file):
    """Generate candidate sample IDs from scan filename"""
    candidates = []
    base = scan_file.replace('.ome.tif.csv', '')
    
    # Candidate 1: Direct (no modifications)
    candidates.append(base)
    
    # Candidate 2: Remove _control_Scan1/Scan2
    if '_control_Scan' in base:
        cand = base.split('_control_Scan')[0]
        candidates.append(cand)
    
    # Candidate 3: Remove all _Scan1, _Scan2 markers
    cand = base.replace('_Scan1', '').replace('_Scan2', '').replace('_control', '')
    candidates.append(cand)
    
    # Candidate 4: Extract text before _Scan pattern (key for most files)
    # This handles: "01264_3H_Scan1_1264_3H_Scan1" -> "01264_3H"
    if '_Scan' in base:
        idx = base.find('_Scan')
        first_part = base[:idx]
        candidates.append(first_part)
    
    # Candidate 5: For files with repeated patterns
    if '_' in base and any(x in base for x in ['_Scan', '-Scan']):
        parts = base.split('_')
        if len(parts) >= 2:
            first_part = parts[0]
            candidates.append(first_part)
            # Also try: first part + next part (in case sample ID has underscore)
            if len(parts) >= 3:
                combined = parts[0] + '_' + parts[1]
                candidates.append(combined)
    
    # Candidate 6: Handle hyphen vs underscore variants
    for cand in list(candidates):
        if '-' in cand:
            candidates.append(cand.replace('-', '_'))
        elif '_' in cand:
            candidates.append(cand.replace('_', '-'))
    
    # Candidate 7: With leading zero prepended (for 4-digit codes)
    for cand in list(candidates):
        if cand.startswith('1') and len(cand.split('-')[0]) == 4:
            candidates.append('0' + cand)
    
    return list(dict.fromkeys(candidates))  # Remove duplicates while preserving order

for scan_file in scan_files:
    candidates = extract_sample_ids(scan_file)
    found = False
    
    for candidate in candidates:
        if candidate in metadata_lookup:
            file_to_sample[scan_file] = candidate
            matched += 1
            found = True
            break
    
    if not found:
        not_matched.append((scan_file, candidates[0]))

print(f"Matched: {matched} files")
print(f"Not matched: {len(not_matched)} files")

if not_matched:
    print(f"\nNot matched:")
    for scan_file, base_name in not_matched:
        print(f"  {scan_file} -> {base_name}")

Found 40 scan CSV files

Matched: 40 files
Not matched: 0 files


In [7]:
# Add metadata columns to each scan CSV file
# Remove any previously mapped columns and add them fresh
processed = 0
errors = []
metadata_columns = ['Batch number', 'Type of cancer', 'Main cancer grouping', 'Main cancer Grouping', 'Mian cancer Grouping', 'Category']

for scan_file, sample_id in file_to_sample.items():
    try:
        # Read the scan file
        scan_df = pd.read_csv(scan_file)
        
        # Remove previously mapped metadata columns if they exist
        scan_df = scan_df.drop(columns=[col for col in metadata_columns if col in scan_df.columns])
        
        # Get metadata for this sample
        meta = metadata_lookup[sample_id]
        
        # Add metadata columns (these will be the same for all rows in the file)
        scan_df['Batch number'] = meta['Batch number']
        scan_df['Type of cancer'] = meta['Type of cancer']
        scan_df['Main cancer grouping'] = meta['Main cancer grouping']
        scan_df['Category'] = meta['Category']
        
        # Save back to file
        scan_df.to_csv(scan_file, index=False)
        processed += 1
        print(f"✓ {scan_file}: added metadata for {sample_id}")
        
    except Exception as e:
        errors.append((scan_file, str(e)))
        print(f"✗ {scan_file}: ERROR - {e}")

print(f"\n" + "="*80)
print(f"Processed: {processed} files")
if errors:
    print(f"Errors: {len(errors)} files")

✓ 01036-1D_Scan1.ome.tif.csv: added metadata for 01036-1D
✓ 01041-2.1_Scan1.ome.tif.csv: added metadata for 01041-2.1
✓ 01059-17B_Scan1.ome.tif.csv: added metadata for 01059-17B
✓ 01059-A_Scan1.ome.tif.csv: added metadata for 01059-A
✓ 01064-1.10_Scan1.ome.tif.csv: added metadata for 01064-1.10
✓ 01064-1B_Scan1_T.ome.tif.csv: added metadata for 01064-1B
✓ 01064-A1_control_Scan1.ome.tif.csv: added metadata for 01064-A1
✓ 01072-1A_Scan1.ome.tif.csv: added metadata for 01072-1A
✓ 01082-1B_Scan2.ome.tif.csv: added metadata for 01082-1B
✓ 01082-1H_Scan1.ome.tif.csv: added metadata for 01082-1H
✓ 01085-1M_Scan2.ome.tif.csv: added metadata for 01085-1M
✓ 01095-1K_Scan1.ome.tif.csv: added metadata for 01095-1K
✓ 01096-08T_Scan1.ome.tif.csv: added metadata for 01096-08T
✓ 01096-17T_Scan1_1096_17T_Scan1.ome.tif.csv: added metadata for 01096-17T
✓ 01096-1C_Scan1.ome.tif.csv: added metadata for 01096-1C
✓ 01102-Scan2_1102-Scan2.ome.tif.csv: added metadata for 01102-Scan2
✓ 01109-1.8_Scan1.ome.tif.

In [8]:
# Verify results - show sample of updated files
print("Verification - Sample of updated scan files:\n")

sample_files = list(file_to_sample.keys())[:3]
for scan_file in sample_files:
    df = pd.read_csv(scan_file)
    print(f"\n{scan_file}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Metadata columns:")
    if 'Batch number' in df.columns:
        print(f"    - Batch number: {df['Batch number'].iloc[0]}")
        print(f"    - Type of cancer: {df['Type of cancer'].iloc[0]}")
        print(f"    - Main cancer grouping: {df['Main cancer grouping'].iloc[0]}")
        print(f"    - Category: {df['Category'].iloc[0]}")

Verification - Sample of updated scan files:


01036-1D_Scan1.ome.tif.csv:
  Shape: (268920, 75)
  Columns: ['Image', 'CellID', 'CentroidX_um', 'CentroidY_um', 'Area_um2', 'Classification', 'ParentAnnotations', 'ContainingAnnotations', 'Geometry_um', 'DAPI: Cell: Mean', 'CD8: Cell: Mean', 'TCR: Cell: Mean', 'GrB: Cell: Mean', 'HLA-DR: Cell: Mean', 'PD-1: Cell: Mean', 'Pan-CK: Cell: Mean', 'Autofluorescence: Cell: Mean', 'Signed distance to annotation Ignore µm', 'Signed distance to annotation Tumour µm', 'Signed distance to annotation Stroma µm', 'Distance to detection HLA-DR µm', 'Distance to detection CD8 cells µm', 'Distance to detection Total conventional cytotoxic effector cells µm', 'Distance to detection Active cytotoxic effector cells µm', 'Distance to detection Other µm', 'Distance to detection Terminally exhausted cells µm', 'Distance to detection CD8+gdT+ cells µm', 'Distance to detection Bystander killer cells µm', 'Distance to detection Tumour µm', 'Distance to detection C

In [9]:
# Summary report
print("\nSUMMARY REPORT")
print("="*80)
print(f"Total scan CSV files: {len(scan_files)}")
print(f"Successfully matched and updated: {processed}")
print(f"Not matched: {len(not_matched)}")
print(f"Processing errors: {len(errors)}")

if not_matched:
    print(f"\nFiles with no metadata match (likely removed from analysis):")
    for scan_file, base_name in not_matched:
        print(f"  - {scan_file}")


SUMMARY REPORT
Total scan CSV files: 40
Successfully matched and updated: 40
Not matched: 0
Processing errors: 0
